
# 01_fit_msaa_save_npz.ipynb

Fit **across-condition Spatial AA** and save outputs as `.npz` files.

Run this first.


In [6]:

K_VALUES = [300, 500, 700]
CONDITIONS_TO_USE = ["intact", "word", "rest"]
TRIM_REST_BY = 100
RNG_SEED = 42

PIEMAN_MAT_PATH = "data/pieman/raw/pieman_data.mat"
POSTERIOR_MAT_PATH = "data/pieman/raw/pieman_posterior_K700.mat"
HELPERS_DIR = "helpers"
HYPERTOOLS_NORMALIZE_PATH = "hypertools/normalize.py"
SAVE_DIR = "msaa_outputs_npz"

MSAA_OPTS = dict(
    maxiter=200,
    conv_crit=1e-6,
    fix_var_iter=5,
    heteroscedastic=False,
    use_gpu=False,
    rngSEED=RNG_SEED,
)


In [7]:

%matplotlib inline
import os, sys, time, importlib.util
import numpy as np, pandas as pd
from scipy.io import loadmat
from sklearn.cluster import KMeans

HT_NORMALIZE_AVAILABLE = False
try:
    spec = importlib.util.spec_from_file_location("normalize", HYPERTOOLS_NORMALIZE_PATH)
    ht = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(ht)
    HT_NORMALIZE_AVAILABLE = True
    print("Loaded hypertools normalize.")
except Exception:
    print("hypertools normalize not available; using fallback normalization.")
    ht = None

sys.path.append(HELPERS_DIR)
from MultiSubject_AA import Subject, multi_subject_aa
os.makedirs(SAVE_DIR, exist_ok=True)


hypertools normalize not available; using fallback normalization.


In [8]:

def array_cutter(data_list, cut_length):
    out = [np.asarray(x).copy() for x in data_list]
    if cut_length <= 0:
        return out
    return [x[:-cut_length] for x in out]

def coerce_TxV(x):
    x = np.asarray(x)
    T, V = x.shape
    if V < T:
        x = x.T
    x = x - x.mean(axis=0, keepdims=True)
    return x.astype(np.float32, copy=False)

def normalize_across_fallback(subject_list, eps=1e-8):
    subj = [coerce_TxV(x) for x in subject_list]
    stacked = np.vstack(subj)
    mu = stacked.mean(axis=0, keepdims=True)
    sd = stacked.std(axis=0, keepdims=True) + eps
    return [((x - mu) / sd).astype(np.float32, copy=False) for x in subj]

def normalize_subject_list(subject_list):
    subj = [coerce_TxV(x) for x in subject_list]
    if HT_NORMALIZE_AVAILABLE:
        try:
            return ht.normalize(subj, normalize="across")
        except Exception:
            return normalize_across_fallback(subj)
    return normalize_across_fallback(subj)

def load_pieman_data():
    pieman_data = loadmat(PIEMAN_MAT_PATH)
    data, conds = [], []
    for c in CONDITIONS_TO_USE:
        next_data = list(map(lambda i: pieman_data[c][:, i][0], np.arange(pieman_data[c].shape[1])))
        data.extend(next_data)
        conds.extend([c] * len(next_data))
    conds_array = np.array(conds)
    condition_data = {}
    for c in CONDITIONS_TO_USE:
        cur = [data[i] for i in np.where(conds_array == c)[0]]
        if c == "rest" and TRIM_REST_BY > 0:
            cur = array_cutter(cur, TRIM_REST_BY)
        condition_data[c] = normalize_subject_list(cur)

    all_subjects, all_condition_labels = [], []
    for c in CONDITIONS_TO_USE:
        for x in condition_data[c]:
            all_subjects.append(x)
            all_condition_labels.append(c)
    return condition_data, np.array(all_condition_labels), all_subjects

condition_data, condition_labels_str, all_subjects = load_pieman_data()
condition_names = sorted(np.unique(condition_labels_str))
condition_to_code = {c: i for i, c in enumerate(condition_names)}
condition_codes = np.array([condition_to_code[c] for c in condition_labels_str])

print("Total subjects:", len(all_subjects))
print(pd.Series(condition_labels_str).value_counts())


Total subjects: 108
intact    36
word      36
rest      36
Name: count, dtype: int64


In [9]:

posterior = loadmat(POSTERIOR_MAT_PATH)
centers = posterior["posterior"]["centers"][0][0][0][0][0]
centers = np.asarray(centers, dtype=float)
if centers.shape[1] != 3 and centers.shape[0] == 3:
    centers = centers.T

widths = 10

network_colors = {
    "Visual": "#781286",
    "Somatomotor": "#4682B4",
    "DorsalAttn": "#00760E",
    "VentralAttn": "#C69C6D",
    "Limbic": "#F3C300",
    "Frontoparietal": "#FF6800",
    "Default": "#A6A6A6",
}

lookup_table = {
    "Visual": 1,
    "Somatomotor": 2,
    "DorsalAttn": 3,
    "VentralAttn": 4,
    "Limbic": 5,
    "Frontoparietal": 6,
    "Default": 7,
}

def build_networks_cmu_from_centers(centers, n_networks=7, seed=42):
    kmeans = KMeans(n_clusters=n_networks, random_state=seed, n_init=10)
    labels = kmeans.fit_predict(centers)
    names = list(lookup_table.keys())
    mapping = {i: names[i] for i in range(n_networks)}
    return pd.DataFrame({
        "x": centers[:, 0],
        "y": centers[:, 1],
        "z": centers[:, 2],
        "network": [mapping[l] for l in labels],
    })

networks_cmu = build_networks_cmu_from_centers(centers)

def node_labels(centers, widths, networks_df):
    coords = networks_df[["x", "y", "z"]].values
    labels = []
    for c in np.asarray(centers):
        dists = np.linalg.norm(coords - c, axis=1)
        idx = np.argmin(dists)
        net = networks_df.iloc[idx]["network"]
        labels.append({
            "x": c[0], "y": c[1], "z": c[2],
            "network": net,
            "code": lookup_table.get(net, 0)
        })
    return pd.DataFrame(labels)

node_codes = node_labels(centers, widths, networks_cmu)


In [10]:

def build_subject_objects_spatial(subject_list):
    subj_objs = []
    for X in subject_list:
        Xtv = coerce_TxV(X)
        subj_objs.append(Subject(X=Xtv, sX=Xtv))
    return subj_objs

def fit_spatial_msaa(subject_list, K, opts=None, verbose=True):
    opts = dict(MSAA_OPTS if opts is None else opts)
    subj_objs = build_subject_objects_spatial(subject_list)
    results_subj, C, cost_fun, varexpl, secs = multi_subject_aa(
        subj_objs,
        noc=K,
        opts=opts,
    )
    if verbose:
        print(f"[Across-condition Spatial AA] K={K} | VarExpl={varexpl*100:.2f}%")
        print("subj0 sXC shape:", np.asarray(results_subj[0]['sXC']).shape)
        print("subj0 S shape:", np.asarray(results_subj[0]['S']).shape)
    return {
        "results_subj": results_subj,
        "C": C,
        "cost_fun": cost_fun,
        "varexpl": varexpl,
        "secs": secs,
        "K": K,
    }

def save_msaa_npz(save_path, fit_res):
    np.savez_compressed(
        save_path,
        K=np.array(fit_res["K"]),
        results_subj=np.array(fit_res["results_subj"], dtype=object),
        C=np.array(fit_res["C"], dtype=object),
        cost_fun=np.array(fit_res["cost_fun"], dtype=object),
        varexpl=np.array(fit_res["varexpl"]),
        secs=np.array(fit_res["secs"]),
        condition_labels_str=np.array(condition_labels_str, dtype=object),
        condition_codes=np.array(condition_codes),
        condition_names=np.array(condition_names, dtype=object),
        centers=np.array(centers),
        widths=np.array(widths),
        networks_cmu=np.array(networks_cmu.to_dict("records"), dtype=object),
        node_codes=np.array(node_codes.to_dict("records"), dtype=object),
        lookup_table_keys=np.array(list(lookup_table.keys()), dtype=object),
        lookup_table_vals=np.array(list(lookup_table.values())),
        network_color_keys=np.array(list(network_colors.keys()), dtype=object),
        network_color_vals=np.array(list(network_colors.values()), dtype=object),
    )

saved_paths = []
for K in K_VALUES:
    fit_res = fit_spatial_msaa(all_subjects, K, opts=MSAA_OPTS, verbose=True)
    save_path = os.path.join(SAVE_DIR, f"spatialAA_acrossCond_K{K}.npz")
    save_msaa_npz(save_path, fit_res)
    saved_paths.append(save_path)
    print("Saved:", save_path)

saved_paths


/opt/homebrew/anaconda3/envs/archs/lib/python3.10/site-packages/numpy/lib/function_base.py:2853: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/opt/homebrew/anaconda3/envs/archs/lib/python3.10/site-packages/numpy/lib/function_base.py:2854: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


[Across-condition Spatial AA] K=300 | VarExpl=76.45%
subj0 sXC shape: (300, 300)
subj0 S shape: (300, 700)
Saved: msaa_outputs_npz/spatialAA_acrossCond_K300.npz


/opt/homebrew/anaconda3/envs/archs/lib/python3.10/site-packages/numpy/lib/function_base.py:2853: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/opt/homebrew/anaconda3/envs/archs/lib/python3.10/site-packages/numpy/lib/function_base.py:2854: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


[Across-condition Spatial AA] K=500 | VarExpl=90.10%
subj0 sXC shape: (300, 500)
subj0 S shape: (500, 700)
Saved: msaa_outputs_npz/spatialAA_acrossCond_K500.npz


/opt/homebrew/anaconda3/envs/archs/lib/python3.10/site-packages/numpy/lib/function_base.py:2853: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/opt/homebrew/anaconda3/envs/archs/lib/python3.10/site-packages/numpy/lib/function_base.py:2854: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


[Across-condition Spatial AA] K=700 | VarExpl=100.00%
subj0 sXC shape: (300, 700)
subj0 S shape: (700, 700)
Saved: msaa_outputs_npz/spatialAA_acrossCond_K700.npz


['msaa_outputs_npz/spatialAA_acrossCond_K300.npz',
 'msaa_outputs_npz/spatialAA_acrossCond_K500.npz',
 'msaa_outputs_npz/spatialAA_acrossCond_K700.npz']